In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Medallion Architecture (Full Pipeline in SQL)
# MAGIC **0. Setup**

# COMMAND ----------
import re
from datetime import datetime

current_user = spark.sql("SELECT current_user()").collect()[0][0]

# Change to your raw file path
# RAW_PATH = f"/Workspace/Users/{current_user}/bda_course/raw_data"
RAW_PATH =f"/Volumes/workspace/default/course_data/raw_data"
CATALOG = "workspace"
SCHEMA = "medallion_sql"

# Initialize Catalog and Schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE {CATALOG}.{SCHEMA}")

print(f"Catalog     : {CATALOG}")
print(f"Schema      : {SCHEMA}")
print(f"Raw path    : {RAW_PATH}")


# COMMAND ----------
# MAGIC %md
# MAGIC ## 1. Bronze Layer (Ingestion)
# MAGIC *Note: We keep Python for Bronze ingestion because reading arbitrary CSVs and dynamically stripping bad characters from column names (`{`, `}`, ` `, etc.) is much cleaner in Python than in pure SQL.*

# COMMAND ----------

def ingest_to_bronze(filename, table_name):
    # 1. Read Raw CSV (Everything as string)
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .option("multiLine", "true")
        .option("escape", '"')
        .csv(f"{RAW_PATH}/{filename}")
    )
    
    # 2. Clean column names (replace spaces and special chars with underscores)
    cleaned_cols = [re.sub(r"[ ,;{}()\n\t=]", "_", c.strip()) for c in df.columns]
    df = df.toDF(*cleaned_cols)
    
    # 3. Register as temp view to process with SQL
    df.createOrReplaceTempView("temp_raw_csv")
    
    # 4. Write to Bronze with metadata using SQL
    target_table = f"{CATALOG}.{SCHEMA}.bronze_{table_name}"
    spark.sql(f"""
        CREATE OR REPLACE TABLE {target_table} AS 
        SELECT 
            *,
            CURRENT_TIMESTAMP() AS _ingest_time,
            '{filename}' AS _source_file,
            CURRENT_DATE() AS _batch_date
        FROM temp_raw_csv
    """)
    
    count = spark.sql(f"SELECT COUNT(*) FROM {target_table}").collect()[0][0]
    print(f"Bronze Table: {target_table:40s} | Rows: {count:,}")

# Ingest all files
ingest_to_bronze("RAW_Product.csv", "product")
ingest_to_bronze("RAW_Date.csv", "date")
ingest_to_bronze("RAW_Customer.csv", "customer")
ingest_to_bronze("RAW_Reseller.csv", "reseller")
ingest_to_bronze("RAW_SalesTerritory.csv", "sales_territory")
ingest_to_bronze("RAW_SalesOrder.csv", "sales_order")
ingest_to_bronze("RAW_Sales.csv", "sales")


In [0]:
# COMMAND ----------
# MAGIC %md
# MAGIC ## 2. Silver Layer (Data Modeling & Cleaning in SQL)

# COMMAND ----------

# --- 1. Dim Customer ---
spark.sql(rf"""
    CREATE OR REPLACE TABLE silver_dim_customer AS
    WITH dedup AS (
        SELECT DISTINCT
            Customer_ID AS CustomerNaturalKey,
            TRIM(Customer) AS CustomerName,
            TRIM(City) AS City,
            TRIM(`State-Province`) AS StateProvince,
            TRIM(`Country-Region`) AS CountryRegion,
            TRIM(Postal_Code) AS PostalCode
        FROM bronze_customer
        WHERE Customer_ID IS NOT NULL
    )
    SELECT 
        ROW_NUMBER() OVER(ORDER BY CustomerNaturalKey) AS CustomerKey,
        CustomerNaturalKey,
        COALESCE(CustomerName, 'Unknown') AS CustomerName,
        COALESCE(City, 'Unknown') AS City,
        INITCAP(StateProvince) AS StateProvince,
        INITCAP(CountryRegion) AS CountryRegion,
        COALESCE(PostalCode, '00000') AS PostalCode
    FROM dedup
""")
print("Created silver_dim_customer")

# --- 2. Dim Product ---
spark.sql(rf"""
    CREATE OR REPLACE TABLE silver_dim_product AS
    WITH dedup AS (
        SELECT SKU, FIRST(Product) as Product, FIRST(Model) as Model, FIRST(Category) as Category, 
               FIRST(Subcategory) as Subcategory, FIRST(Color) as Color, 
               FIRST(List_Price) as List_Price, FIRST(Standard_Cost) as Standard_Cost
        FROM bronze_product
        WHERE SKU IS NOT NULL
        GROUP BY SKU
    )
    SELECT 
        ROW_NUMBER() OVER(ORDER BY SKU) AS ProductKey,
        SKU,
        TRIM(Product) AS Product,
        TRIM(Model) AS Model,
        COALESCE(INITCAP(TRIM(Category)), 'Unknown') AS Category,
        INITCAP(TRIM(Subcategory)) AS Subcategory,
        COALESCE(INITCAP(TRIM(Color)), 'No Color') AS Color,
        CAST(REGEXP_REPLACE(List_Price, '[^\\d\\.\\-]', '') AS DOUBLE) AS ListPrice,
        COALESCE(CAST(REGEXP_REPLACE(Standard_Cost, '[^\\d\\.\\-]', '') AS DOUBLE), 0.0) AS StandardCost
    FROM dedup
    WHERE CAST(REGEXP_REPLACE(List_Price, '[^\\d\\.\\-]', '') AS DOUBLE) > 0
""")
print("Created silver_dim_product")

# --- 3. Dim Reseller ---
spark.sql(rf"""
    CREATE OR REPLACE TABLE silver_dim_reseller AS
    WITH dedup AS (
        SELECT DISTINCT
            Reseller_ID AS ResellerNaturalKey,
            TRIM(Reseller) AS Reseller,
            TRIM(Business_Type) AS BusinessType,
            TRIM(City) AS City,
            TRIM(`State-Province`) AS StateProvince,
            TRIM(`Country-Region`) AS CountryRegion,
            TRIM(Postal_Code) AS PostalCode
        FROM bronze_reseller
        WHERE Reseller_ID IS NOT NULL
    )
    SELECT 
        ROW_NUMBER() OVER(ORDER BY ResellerNaturalKey) AS ResellerKey,
        ResellerNaturalKey,
        Reseller,
        COALESCE(INITCAP(BusinessType), 'Unknown') AS BusinessType,
        COALESCE(City, 'Unknown') AS City,
        INITCAP(StateProvince) AS StateProvince,
        INITCAP(CountryRegion) AS CountryRegion,
        COALESCE(PostalCode, '00000') AS PostalCode
    FROM dedup
""")
print("Created silver_dim_reseller")

# --- 4. Dim Territory ---
spark.sql(rf"""
    CREATE OR REPLACE TABLE silver_dim_territory AS
    WITH dedup AS (
        SELECT DISTINCT TRIM(Region) AS Region, TRIM(Country) AS Country, TRIM(Sales_Group) AS SalesGroup
        FROM bronze_sales_territory
        WHERE Region IS NOT NULL AND Country IS NOT NULL
    )
    SELECT 
        ROW_NUMBER() OVER(ORDER BY Region) AS SalesTerritoryKey,
        INITCAP(Region) AS Region,
        INITCAP(Country) AS Country,
        INITCAP(SalesGroup) AS SalesGroup
    FROM dedup
""")
print("Created silver_dim_territory")

# --- 5. Dim Date ---
spark.sql(rf"""
    CREATE OR REPLACE TABLE silver_dim_date AS
    WITH parsed AS (
        SELECT DISTINCT
            COALESCE(
                TRY_TO_DATE(Date, 'yyyy-MM-dd'),
                TRY_TO_DATE(Date, 'MM/dd/yyyy'),
                TRY_TO_DATE(Date, 'dd-MM-yyyy'),
                TRY_TO_DATE(Date, 'yyyyMMdd'),
                TRY_TO_DATE(Date, 'MMM dd, yyyy')
            ) AS Date,
            Fiscal_Year, Fiscal_Quarter, Month
        FROM bronze_date
        WHERE Date IS NOT NULL
    )
    SELECT 
        CAST(DATE_FORMAT(Date, 'yyyyMMdd') AS INT) AS DateKey,
        Date,
        CONCAT('FY', REGEXP_EXTRACT(Fiscal_Year, '(\\d{{4}})', 1)) AS FiscalYear,
        COALESCE(Fiscal_Quarter, CONCAT('FY', REGEXP_EXTRACT(Fiscal_Year, '(\\d{{4}})', 1), ' Q', CAST(CEIL((MONTH(Date) + CASE WHEN MONTH(Date) >= 7 THEN -6 ELSE 6 END) / 3.0) AS INT))) AS FiscalQuarter,
        Month,
        CAST(DATE_FORMAT(Date, 'yyyyMM') AS INT) AS MonthKey
    FROM parsed
    WHERE Date IS NOT NULL
""")
print("Created silver_dim_date")

# --- 6. Dim Sales Order ---
spark.sql(rf"""
    CREATE OR REPLACE TABLE silver_dim_salesorder AS
    WITH dedup AS (
        SELECT DISTINCT TRIM(Sales_Order) AS SalesOrder, TRIM(Sales_Order_Line) AS SalesOrderLine, COALESCE(INITCAP(TRIM(Channel)), 'Unknown') AS Channel
        FROM bronze_sales_order
        WHERE Sales_Order_Line IS NOT NULL
    )
    SELECT 
        CAST(REGEXP_EXTRACT(SalesOrder, 'SO(\\d+)', 1) AS BIGINT) * 100 + CAST(REGEXP_EXTRACT(SalesOrderLine, '-\\s*(\\d+)$', 1) AS BIGINT) AS SalesOrderLineKey,
        SalesOrder,
        SalesOrderLine,
        Channel
    FROM dedup
""")
print("Created silver_dim_salesorder")

# --- 7. Fact Sales (Main Join) ---
spark.sql(rf"""
    CREATE OR REPLACE TABLE silver_fact_sales AS
    WITH cleaned_sales AS (
        SELECT 
            -- Dates
            COALESCE(TRY_TO_DATE(Order_Date, 'yyyy-MM-dd'), TRY_TO_DATE(Order_Date, 'MM/dd/yyyy')) AS OrderDate,
            COALESCE(TRY_TO_DATE(Due_Date, 'yyyy-MM-dd'), TRY_TO_DATE(Due_Date, 'MM/dd/yyyy')) AS DueDate,
            COALESCE(TRY_TO_DATE(Ship_Date, 'yyyy-MM-dd'), TRY_TO_DATE(Ship_Date, 'MM/dd/yyyy')) AS ShipDate,
            
            -- Numbers
            CAST(ROUND(CAST(Order_Quantity AS DOUBLE), 0) AS INT) AS OrderQuantity,
            CAST(Unit_Price AS DOUBLE) AS UnitPrice,
            CASE WHEN Unit_Price_Discount_Pct LIKE '%\\%%' THEN CAST(REGEXP_EXTRACT(Unit_Price_Discount_Pct, '([\\d\\.]+)%', 1) AS DOUBLE) / 100.0 ELSE CAST(Unit_Price_Discount_Pct AS DOUBLE) END AS UnitPriceDiscountPct,
            CAST(Product_Standard_Cost AS DOUBLE) AS ProductStandardCost,
            CAST(Total_Product_Cost AS DOUBLE) AS TotalProductCost,
            CAST(Extended_Amount AS DOUBLE) AS ExtendedAmount,
            CAST(Sales_Amount AS DOUBLE) AS SalesAmount,
            
            -- Join Keys
            Customer_ID, SKU, Reseller_ID, INITCAP(TRIM(Region)) AS Region, INITCAP(TRIM(Country)) AS Country, TRIM(Sales_Order_Line) AS SalesOrderLine, INITCAP(TRIM(Channel)) AS Channel
        FROM bronze_sales
    ),
    deduped_sales AS (
        SELECT * FROM (
            SELECT *, ROW_NUMBER() OVER(PARTITION BY SalesOrderLine ORDER BY OrderDate DESC) as rn FROM cleaned_sales
        ) WHERE rn = 1
    )
    SELECT 
        so.SalesOrderLineKey,
        c.CustomerKey,
        p.ProductKey,
        r.ResellerKey,
        t.SalesTerritoryKey,
        CAST(DATE_FORMAT(s.OrderDate, 'yyyyMMdd') AS INT) AS OrderDateKey,
        s.Channel,
        s.OrderQuantity,
        s.UnitPrice,
        s.UnitPriceDiscountPct,
        s.ProductStandardCost,
        s.TotalProductCost,
        s.ExtendedAmount,
        s.SalesAmount,
        s.OrderDate,
        s.DueDate,
        s.ShipDate,
        CASE WHEN s.SalesAmount < 0 THEN 1 ELSE 0 END AS IsReturn
    FROM deduped_sales s
    LEFT JOIN silver_dim_customer c ON s.Customer_ID = c.CustomerNaturalKey
    LEFT JOIN silver_dim_product p ON s.SKU = p.SKU
    LEFT JOIN silver_dim_reseller r ON s.Reseller_ID = r.ResellerNaturalKey
    LEFT JOIN silver_dim_territory t ON s.Region = t.Region AND s.Country = t.Country
    LEFT JOIN silver_dim_salesorder so ON s.SalesOrderLine = so.SalesOrderLine
""")
print("Created silver_fact_sales")

In [0]:




# COMMAND ----------
# MAGIC %md
# MAGIC ## 3. Gold Layer (Business Aggregations in SQL)

# COMMAND ----------

# --- 1. Sales by Month ---
spark.sql(rf"""
    CREATE OR REPLACE TABLE gold_sales_by_month AS
    WITH base AS (
        SELECT 
            d.FiscalYear, d.FiscalQuarter, d.MonthKey, t.Region, t.Country, t.SalesGroup, p.Category, f.Channel,
            COUNT(f.SalesOrderLineKey) AS OrderLines,
            SUM(f.OrderQuantity) AS TotalUnits,
            ROUND(SUM(f.SalesAmount), 2) AS TotalRevenue,
            ROUND(AVG(f.SalesAmount), 2) AS AvgOrderValue,
            ROUND(SUM(f.TotalProductCost), 2) AS TotalCost,
            ROUND(SUM(f.SalesAmount) - SUM(f.TotalProductCost), 2) AS GrossProfit,
            COUNT(DISTINCT f.CustomerKey) AS UniqueCustomers
        FROM silver_fact_sales f
        JOIN silver_dim_date d ON f.OrderDateKey = d.DateKey
        LEFT JOIN silver_dim_territory t ON f.SalesTerritoryKey = t.SalesTerritoryKey
        LEFT JOIN silver_dim_product p ON f.ProductKey = p.ProductKey
        WHERE f.IsReturn = 0
        GROUP BY 1, 2, 3, 4, 5, 6, 7, 8
    )
    SELECT *,
        CASE WHEN TotalRevenue != 0 THEN ROUND((GrossProfit / TotalRevenue) * 100, 1) END AS GrossMarginPct,
        LAG(TotalRevenue) OVER(PARTITION BY Region, Category, Channel ORDER BY MonthKey) AS PrevMonthRevenue,
        CASE WHEN LAG(TotalRevenue) OVER(PARTITION BY Region, Category, Channel ORDER BY MonthKey) != 0 
             THEN ROUND((TotalRevenue - LAG(TotalRevenue) OVER(PARTITION BY Region, Category, Channel ORDER BY MonthKey)) / LAG(TotalRevenue) OVER(PARTITION BY Region, Category, Channel ORDER BY MonthKey) * 100, 1)
        END AS MoMGrowthPct,
        CURRENT_TIMESTAMP() AS _gold_timestamp
    FROM base
""")
print("Created gold_sales_by_month")

# --- 2. Product Ranking ---
spark.sql(rf"""
    CREATE OR REPLACE TABLE gold_product_ranking AS
    WITH total_rev AS (SELECT SUM(SalesAmount) AS GrandTotal FROM silver_fact_sales WHERE IsReturn = 0),
    product_agg AS (
        SELECT 
            p.ProductKey, p.SKU, p.Product, p.Model, p.Category, p.Subcategory, p.Color,
            SUM(f.OrderQuantity) AS UnitsSold,
            ROUND(SUM(f.SalesAmount), 2) AS TotalRevenue,
            ROUND(SUM(f.TotalProductCost), 2) AS TotalCost,
            ROUND(AVG(f.UnitPrice), 2) AS AvgSellingPrice,
            COUNT(f.SalesOrderLineKey) AS OrderLines,
            COUNT(DISTINCT f.CustomerKey) AS UniqueCustomers
        FROM silver_fact_sales f
        LEFT JOIN silver_dim_product p ON f.ProductKey = p.ProductKey
        WHERE f.IsReturn = 0
        GROUP BY 1, 2, 3, 4, 5, 6, 7
    )
    SELECT a.*,
        ROUND(a.TotalRevenue - a.TotalCost, 2) AS GrossProfit,
        CASE WHEN a.TotalRevenue != 0 THEN ROUND(((a.TotalRevenue - a.TotalCost) / a.TotalRevenue) * 100, 1) END AS GrossMarginPct,
        CASE WHEN tr.GrandTotal != 0 THEN ROUND((a.TotalRevenue / tr.GrandTotal) * 100, 2) END AS RevSharePct,
        RANK() OVER(ORDER BY a.TotalRevenue DESC) AS RevenueRank,
        RANK() OVER(PARTITION BY a.Category ORDER BY a.TotalRevenue DESC) AS RevenueRankInCategory,
        CURRENT_TIMESTAMP() AS _gold_timestamp
    FROM product_agg a CROSS JOIN total_rev tr
""")
print("Created gold_product_ranking")

# --- 3. Customer Summary (RFM Model) ---
spark.sql(rf"""
    CREATE OR REPLACE TABLE gold_customer_summary AS
    WITH rfm_base AS (
        SELECT 
            CustomerKey,
            DATEDIFF(CURRENT_DATE(), MAX(OrderDate)) AS RecencyDays,
            COUNT(SalesOrderLineKey) AS Frequency,
            ROUND(SUM(SalesAmount), 2) AS LifetimeValue,
            ROUND(AVG(SalesAmount), 2) AS AvgOrderValue,
            ROUND(SUM(TotalProductCost), 2) AS TotalCost,
            MAX(OrderDate) AS LastOrderDate,
            MIN(OrderDate) AS FirstOrderDate,
            COUNT(DISTINCT ProductKey) AS ProductVariety
        FROM silver_fact_sales
        WHERE IsReturn = 0 AND CustomerKey IS NOT NULL
        GROUP BY CustomerKey
    ),
    rfm_scores AS (
        SELECT *,
            CASE WHEN RecencyDays <= 30 THEN 5 WHEN RecencyDays <= 90 THEN 4 WHEN RecencyDays <= 180 THEN 3 WHEN RecencyDays <= 365 THEN 2 ELSE 1 END AS R_Score,
            CASE WHEN Frequency >= 20 THEN 5 WHEN Frequency >= 10 THEN 4 WHEN Frequency >= 5 THEN 3 WHEN Frequency >= 2 THEN 2 ELSE 1 END AS F_Score,
            CASE WHEN LifetimeValue >= 20000 THEN 5 WHEN LifetimeValue >= 10000 THEN 4 WHEN LifetimeValue >= 3000 THEN 3 WHEN LifetimeValue >= 500 THEN 2 ELSE 1 END AS M_Score
        FROM rfm_base
    ),
    rfm_segments AS (
        SELECT *, (R_Score + F_Score + M_Score) AS RFM_Score,
            CASE 
                WHEN (R_Score + F_Score + M_Score) >= 13 THEN 'Champions'
                WHEN (R_Score + F_Score + M_Score) >= 10 THEN 'Loyal'
                WHEN (R_Score + F_Score + M_Score) >= 7 THEN 'Potential'
                WHEN (R_Score + F_Score + M_Score) >= 5 THEN 'At Risk'
                ELSE 'Lost' END AS CustomerSegment,
            ROUND(LifetimeValue - TotalCost, 2) AS GrossProfit
        FROM rfm_scores
    )
    SELECT 
        c.CustomerKey, c.CustomerNaturalKey, c.CustomerName, c.City, c.StateProvince, c.CountryRegion,
        COALESCE(r.RecencyDays, 9999) AS RecencyDays, COALESCE(r.Frequency, 0) AS Frequency, COALESCE(r.LifetimeValue, 0.0) AS LifetimeValue,
        r.AvgOrderValue, r.TotalCost, r.LastOrderDate, r.FirstOrderDate, r.ProductVariety,
        r.R_Score, r.F_Score, r.M_Score, r.RFM_Score,
        COALESCE(r.CustomerSegment, 'Lost') AS CustomerSegment,
        COALESCE(r.GrossProfit, 0.0) AS GrossProfit,
        CURRENT_TIMESTAMP() AS _gold_timestamp
    FROM silver_dim_customer c
    LEFT JOIN rfm_segments r ON c.CustomerKey = r.CustomerKey
""")
print("Created gold_customer_summary")

# --- 4. Channel Compare ---
spark.sql(rf"""
    CREATE OR REPLACE TABLE gold_channel_compare AS
    SELECT 
        d.FiscalYear, f.Channel, p.Category,
        COUNT(f.SalesOrderLineKey) AS OrderLines,
        SUM(f.OrderQuantity) AS UnitsSold,
        ROUND(SUM(f.SalesAmount), 2) AS TotalRevenue,
        ROUND(AVG(f.SalesAmount), 2) AS AvgOrderValue,
        ROUND(SUM(f.TotalProductCost), 2) AS TotalCost,
        COUNT(DISTINCT f.CustomerKey) AS UniqueCustomers,
        ROUND(SUM(f.SalesAmount) - SUM(f.TotalProductCost), 2) AS GrossProfit,
        CASE WHEN SUM(f.SalesAmount) != 0 THEN ROUND((SUM(f.SalesAmount) - SUM(f.TotalProductCost)) / SUM(f.SalesAmount) * 100, 1) END AS GrossMarginPct,
        CASE WHEN COUNT(DISTINCT f.CustomerKey) != 0 THEN ROUND(SUM(f.SalesAmount) / COUNT(DISTINCT f.CustomerKey), 2) END AS RevenuePerCustomer,
        CURRENT_TIMESTAMP() AS _gold_timestamp
    FROM silver_fact_sales f
    JOIN silver_dim_date d ON f.OrderDateKey = d.DateKey
    LEFT JOIN silver_dim_product p ON f.ProductKey = p.ProductKey
    WHERE f.IsReturn = 0
    GROUP BY 1, 2, 3
    ORDER BY 1, 2, 3
""")
print("Created gold_channel_compare")


# COMMAND ----------
# MAGIC %md
# MAGIC ## 4. Final Output Summaries

# COMMAND ----------

# MAGIC %sql
# MAGIC SELECT 'Bronze' AS Layer, COUNT(*) as Tables FROM information_schema.tables WHERE table_schema = 'medallion' AND table_name LIKE 'bronze_%'
# MAGIC UNION ALL
# MAGIC SELECT 'Silver' AS Layer, COUNT(*) as Tables FROM information_schema.tables WHERE table_schema = 'medallion' AND table_name LIKE 'silver_%'
# MAGIC UNION ALL
# MAGIC SELECT 'Gold' AS Layer, COUNT(*) as Tables FROM information_schema.tables WHERE table_schema = 'medallion' AND table_name LIKE 'gold_%';

# COMMAND ----------

display(spark.sql("SELECT * FROM gold_channel_compare LIMIT 10"))